**GUIDE**

The goal of `fastllm` library which we are currently refactoring module by module here in these dialogs is to create a unified llm api for OpenAI responses, OpenAI chat, Anthropic and Gemini apis. Users should be able to use this unified / canonical library to be able to swap models/providers effortlessly without needing to change any code.

Refactored dialogs so far:

- `aai-ws/fastllm/nbs/00_types` exists is to create a canonical internal representations across model providers so that fastllm can be a unified api where users can swap models without changing code during a chat session. 
- `aai-ws/fastllm/nbs/01_normalize` exists to convert model responses or stream events into normalized internal representations.
- `aai-ws/fastllm/nbs/02_streaming` provides lossless stream collation (`acollect_stream`) that collects fragmented Delta streams into `StreamSummary` with a `Msg` matching the shape of `Completion.message`. Covers tool call assembly, thinking/text interleaving, and server tool result preservation across all four providers.


You can find the pre-refactor and fully functional snapshot of this library `~/aai-ws/fastllm/fastllm_py/*.py`. The already refactored modules might differ from their counterparts in `fastllm_py/types.py` and `fastllm_py/normalize.py`, as refactoring involves simplifying code, refactoring the code to fastai / literate programming style, redesigning parts, and fixing issues/bugs.


 When users continue the chat with the same provider raw data is used for constructing the provider native input with all required fields to keep the cache intact. When switching over to providers, canonical data is transformed into provider counterpart with best effort.

## Changes Summary (so far)

*This pinned note serves as a persistent reference for SolveitAI context, since earlier prompts and tool results get truncated in chat history. It captures all changes made so far so we can continue without losing track.*

### Goal
Align `StreamSummary` with `Completion` so both produce consistent `Msg` objects with properly ordered `Part`s, and add canonical thinking/reasoning support across all providers.

### 1. `Delta` type (`00_types`)
- Added `thinking: str = ""` field — parallel to `text`, so streaming consumers distinguish reasoning from output text
- Added `model` and `provider` fields

### 2. `StreamSummary` (`02_streaming`)
- Added `model: str` and `provider: str` fields
- Added `message: Msg` field — built from collated text + thinking + tool_calls, mirroring `Completion.message`
- Removed `text` as a top-level field (it's now inside `message.content` as parts)

### 3. `acollect_stream` (`02_streaming`) — order-preserving part assembly
- Builds `parts` list in true stream order using `_append_part` (merges consecutive same-type text/thinking) and `_reserve_tool` (inserts `None` placeholders for tool_use at first-appearance position)
- After stream ends, placeholders are filled with finalized `Part(type="tool_use", data=dict(id, name, arguments, **extra))`
- Handles interleaved thinking/text correctly (each type-switch starts a new Part)
- Now yields `{'text': chunk}` or `{'thinking': chunk}` dicts for incremental streaming
- Final yield is the completed `StreamSummary`

### 4. `ToolCall.extra` preservation (`02_streaming`)
- `_ToolBuf` now has `extra: dict` field
- `acollect_stream` merges `tc.extra` into buffer during streaming
- `_final_tool_calls` passes `extra=tb.extra` through to final `ToolCall` objects
- Tool_use `Part.data` spreads `**tc.extra` so shape matches completion normalizers

### 5. Stream event normalizers (`01_normalize`) — `Delta(thinking=...)`
- **Anthropic**: `thinking_delta` → `Delta(thinking=...)` instead of `Delta(text=...)`
- **OpenAI Responses**: Added `response.reasoning_text.delta` → `Delta(thinking=...)`
- **OpenAI Chat**: Check `reasoning_content` first → thinking-only Delta; otherwise text Delta
- **Gemini**: Check `thought: true` parts first → thinking-only Delta; otherwise text Delta

### 6. Completion normalizers (`01_normalize`) — `Part(type='thinking')`
- **Anthropic**: Already worked via `_ANT_PART` mapping
- **OpenAI Responses**: `_openai_responses_parts` handles `reasoning` output items → `Part(type='thinking')` from summary blocks
- **OpenAI Chat**: `normalize_openai_chat_completion` extracts `reasoning_content` → prepends `Part(type='thinking')`
- **Gemini**: `_gem_part_type` + `normalize_gemini_generate` exported — maps `thought: true` → `Part(type='thinking')`

### 7. Server tool result support (`01_normalize`, `02_streaming`)
- **`_ant_part_type`** (`01_normalize`): `*_tool_result` types now map to `'server_tool_result'` instead of `'tool_result'` — completions produce `Part(type='server_tool_result', data=<raw content block>)`
- **`acollect_stream`** (`02_streaming`): detects `content_block_start` events ending in `_tool_result` and appends `Part(type='server_tool_result', data=<raw content block>)` — streaming now preserves server tool results (e.g. `web_search_tool_result` with encrypted content needed for multi-turn chat)

### 8. Server/MCP tool call streaming fixes (`01_normalize`, `02_streaming`)
- **`_prime_tool_from_raw`** (`02_streaming`): broadened `content_block_start` check from `== "tool_use"` to `endswith("tool_use")` — `server_tool_use` and `mcp_tool_use` are now properly primed with `idx_to_key` mapping, fixing collation (was producing 2 separate tool calls instead of 1)
- **`normalize_anthropic_event`** (`01_normalize`): added `server=b.get("type")!="tool_use"` to ToolCall in `content_block_start` handler — streaming Deltas now carry `server=True` for server tools
- **`_ToolBuf`** (`02_streaming`): added `server: bool = False` field
- **`acollect_stream`** (`02_streaming`): propagates `tc.server` into `_ToolBuf` (sticky True)
- **`_final_tool_calls`** (`02_streaming`): passes `server=tb.server` through to final `ToolCall` objects

### Known gaps
- **Text `Part.data`**: Streaming text parts have `data=None` while completion text parts carry provider metadata (annotations, citations). Deferred — not needed for chat continuation.
- **`cache_creation`** missing in Anthropic streaming usage (present in completion usage but not streaming)


## Server Tool Results & Cross-Provider Web Search — Findings

### Anthropic Server Tools (web_search, web_fetch, etc.)

**Response structure:** In an assistant message, `server_tool_use` blocks are immediately followed by their `*_tool_result` blocks (e.g. `web_search_tool_result`) in the same `content` array. Text blocks with citations follow after. The `*_tool_result` content contains `encrypted_content` and `encrypted_index` blobs that must be passed back verbatim for multi-turn citation continuity.

**Streaming:** The `web_search_tool_result` arrives fully formed in a single `content_block_start` event — no subsequent deltas. Currently missing from our streaming path (not surfaced in `normalize_anthropic_event`).

**ID convention:** Server tool IDs use `srvtoolu_` prefix (vs `toolu_` for regular tools). Litellm uses this prefix as the heuristic for reconstruction.

### Anthropic Message Format Rules

1. Every `tool_use` block in an assistant message **must** have a corresponding `tool_result` in the next user message — no exceptions
2. `server_tool_use` + `*_tool_result` live together in the **assistant** content array (no user `tool_result` needed)
3. Anthropic does NOT distinguish between "web search" and "regular" tool_use by name — only by the `srvtoolu_` prefix and the presence of the result block

### Cross-Provider Swapping (OpenAI → Anthropic)

**Problem:** OpenAI treats web search as a regular tool (`web_search_call` with `ws_` prefixed id). When converting this history to Anthropic format, Anthropic sees a `tool_use` without a `tool_result` and returns 400.

**Solution (tested & working):** Convert the OpenAI web search tool_use into:
1. Assistant message: `[{"type": "tool_use", "id": "...", "name": "web_search", "input": {...}}]`
2. User message: `[{"type": "tool_result", "tool_use_id": "...", "content": "<extracted text>"}]`  
3. Assistant message: `[{"type": "text", "text": "<original response>"}]`

This works both with and without Anthropic's `web_search` tool enabled. Claude uses the tool result as context and responds naturally.

### Cross-Provider Swapping (Anthropic → Other)

When sending Anthropic history to OpenAI/Gemini, `server_tool_use` + `web_search_tool_result` blocks should be stripped or converted to plain text — other providers don't understand them. The text content with citations can be kept as-is (citations become inert text).

### Litellm Approach (reference)

- Stores `web_search_tool_result` blocks in `provider_specific_fields["web_search_results"]` on the message
- On reconstruction for Anthropic: checks `srvtoolu_` prefix → emits `server_tool_use` + matches result by `tool_use_id`
- Streaming: captures full `web_search_tool_result` from `content_block_start` event into accumulator
- `input_json_delta` events are only treated as tool call args when `current_content_block_type` is `tool_use` or `server_tool_use` (not for `*_tool_result`)

# Streaming

> Streaming helpers for lossless event collation.

In [ ]:
#| default_exp streaming

## Imports

In [ ]:
#| export
from dataclasses import dataclass, field, fields
from fastcore.meta import delegates
from fastspec.errors import *
from fastllm.types import *
from fastllm.normalize import *

In [ ]:
#| hide
import yaml,json
from fastspec.spec import *
from fastspec.oapi import *
from fastcore.test import *
from cachy.core import enable_cachy, disable_cachy, doms

In [ ]:
enable_cachy(doms=(*doms,'api.moonshot.ai'), hdrs=('content-type',))

In [ ]:
from lisette.core import lite_mk_func

In [ ]:
specs_path = Path('../specs/')

ant_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'anthropic.yml').read_text())))
oai_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'openai.with-code-samples.yml').read_text())))
gem_spec  = SpecParser.from_discovery(dict2obj(json.loads((specs_path/'gemini.json').read_text())))

In [ ]:
kimi_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['MOONSHOT_API_KEY']}"})
for op in kimi_cli.ops: op.base_url = 'https://api.moonshot.ai/v1'

In [ ]:
ant_spec, oai_spec, gem_spec

(SpecParser(base_url='https://api.anthropic.com', ops=47),
 SpecParser(base_url='https://api.openai.com/v1', ops=241),
 SpecParser(base_url='https://generativelanguage.googleapis.com/', ops=79))

In [ ]:
ant_cli = OpenAPIClient(ant_spec, headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"], "anthropic-version": "2023-06-01"})
oai_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"})
gem_cli = OpenAPIClient(gem_spec, headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]})

In [ ]:
ant_cli.groups.keys()

dict_keys(['messages', 'complete', 'models', 'files', 'skills', 'messages_beta_true', 'models_beta_true', 'files_beta_true', 'skills_beta_true'])

In [ ]:
oai_cli.groups.keys()

dict_keys(['assistants', 'audio', 'batches', 'chat', 'completions', 'containers', 'conversations', 'embeddings', 'evals', 'files', 'fine_tuning', 'images', 'models', 'moderations', 'organization', 'projects', 'realtime', 'responses', 'threads', 'uploads', 'vector_stores', 'videos', 'skills', 'chatkit'])

In [ ]:
gem_cli.groups.keys()

dict_keys(['batches', 'models', 'tuned_models', 'dynamic', 'cached_contents', 'media', 'files', 'generated_files', 'file_search_stores', 'corpora'])

In [ ]:
gem_cli.models

- models.generate_content(model, contents, system_instruction, tools, tool_config, safety_settings, generation_config, cached_content, service_tier, store): *Generates a model response given an input `GenerateContentRequest`. Refer to the [text generation guide](https://ai.google.dev/gemini-api/docs/text-generation) for detailed usage information. Input capabilities differ between models, including tuned models. Refer to the [model guide](https://ai.google.dev/gemini-api/docs/models/gemini) and [tuning guide](https://ai.google.dev/gemini-api/docs/model-tuning) for details.*
- models.generate_answer(model, contents, answer_style, inline_passages, semantic_retriever, safety_settings, temperature): *Generates a grounded answer from the model given an input `GenerateAnswerRequest`.*
- models.stream_generate_content(model, contents, system_instruction, tools, tool_config, safety_settings, generation_config, cached_content, service_tier, store): *Generates a [streamed response](https://ai.google.dev/gemini-api/docs/text-generation?lang=python#generate-a-text-stream) from the model given an input `GenerateContentRequest`.*
- models.embed_content(model, content, task_type, title, output_dimensionality): *Generates a text embedding vector from the input `Content` using the specified [Gemini Embedding model](https://ai.google.dev/gemini-api/docs/models/gemini#text-embedding).*
- models.batch_embed_contents(model, requests): *Generates multiple embedding vectors from the input `Content` which consists of a batch of strings represented as `EmbedContentRequest` objects.*
- models.count_tokens(model, contents, generate_content_request): *Runs a model's tokenizer on input `Content` and returns the token count. Refer to the [tokens guide](https://ai.google.dev/gemini-api/docs/tokens) to learn more about tokens.*
- models.batch_generate_content(model, batch): *Enqueues a batch of `GenerateContent` requests for batch processing.*
- models.async_batch_embed_content(model, batch): *Enqueues a batch of `EmbedContent` requests for batch processing. We have a `BatchEmbedContents` handler in `GenerativeService`, but it was synchronized. So we name this one to be `Async` to avoid confusion.*
- models.generate_message(model, prompt, temperature, candidate_count, top_p, top_k): *Generates a response from the model given an input `MessagePrompt`.*
- models.count_message_tokens(model, prompt): *Runs a model's tokenizer on a string and returns the token count.*
- models.get(name): *Gets information about a specific `Model` such as its version number, token limits, [parameters](https://ai.google.dev/gemini-api/docs/models/generative-models#model-parameters) and other metadata. Refer to the [Gemini models guide](https://ai.google.dev/gemini-api/docs/models/gemini) for detailed model information.*
- models.list(page_size, page_token): *Lists the [`Model`s](https://ai.google.dev/gemini-api/docs/models/gemini) available through the Gemini API.*
- models.predict(model, instances, parameters): *Performs a prediction request.*
- models.predict_long_running(model, instances, parameters): *Same as Predict but returns an LRO.*
- models.generate_text(model, prompt, temperature, candidate_count, max_output_tokens, top_p, top_k, safety_settings, stop_sequences): *Generates a response from the model given an input message.*
- models.embed_text(model, text): *Generates an embedding from the model given an input message.*
- models.batch_embed_text(model, texts, requests): *Generates multiple embeddings from the model given input text in a synchronous call.*
- models.count_text_tokens(model, prompt): *Runs a model's tokenizer on a text and returns the token count.*

## Purpose And Design

`streaming.py` provides lossless stream collation (`acollect_stream`) over canonical deltas

### Delta

Canonical streaming event fragment

In [ ]:
#| export
@dataclass
class Delta:
    "Normalized streaming delta event."
    model: str
    provider: str = None
    text: str = ""
    thinking: str = ""
    refusal: str = ""
    tool_calls: List[ToolCall] = field(default_factory=list)
    server_tool_result: dict = None
    finish_reason: str = None
    usage: Usage = None
    raw: dict = field(default_factory=dict)

In [ ]:
import json
from lisette.core import lite_mk_func

In [ ]:
def simple_add(a:int,b:int):
    'add numbers'
    return a + b

In [ ]:
sch = lite_mk_func(simple_add);sch

{'type': 'function',
 'function': {'name': 'simple_add',
  'description': 'add numbers',
  'parameters': {'type': 'object',
   'properties': {'a': {'description': '', 'type': 'integer'},
    'b': {'description': '', 'type': 'integer'}},
   'required': ['a', 'b']}}}

In [ ]:
async def norm_and_yield(resp, norm_func):
    async for ev in resp:
        d = norm_func(ev)
        if d is not None: yield d

class PrintStream:
    max_print = 10
    cnt = 0
    def print_stream(self,o):
        if not isinstance(o, Completion): 
            if o.get('thinking') and self.cnt<self.max_print: print('🧠',end='',flush=True)
            if (txt:=o.get('text')): print(txt,end='',flush=True)
            self.cnt+=1
        else: self.cnt = 0

In [ ]:
print_stream = PrintStream().print_stream

`ToolCall` normalizes a tool call with `id`, `name`, and `arguments`, anything extra goes into `extra`

`tool_use` parts are built from tool calls, and store `ToolCall` fields in `data`, which will be converted into correct message type based on provider, and allow users to pass `Msg` directly to the chat history

### OpenAI Responses Events

OpenAI responses api returns collated responses in the last stream delta, so we can build the final `Completion` object using the `normalize_openai_response` function without needing anything else:

In [ ]:
#| export
@delegates(model_and_provider)
def normalize_openai_response_event(ev, **kwargs):
    "Normalize OpenAI Responses API stream event into Delta."
    kwargs['model'],kwargs['provider'] = kwargs.get('model',None), kwargs.get('provider',None)
    typ = ev.get("type")
    if typ == "response.output_text.delta":            return Delta(text=ev.get("delta"), raw=ev, **kwargs)
    if typ == "response.reasoning_text.delta":         return Delta(thinking=ev.get("delta",""), raw=ev, **kwargs)
    if typ == "response.reasoning_summary_text.delta": return Delta(thinking=ev.get("delta",""), raw=ev, **kwargs)
    if typ == "response.completed":                    return Delta(raw=ev, **kwargs)
    if typ == "error": raise api_error_from_event(ev, provider="openai", endpoint="responses.stream")

In [ ]:
#| export
async def _acollect_stream_openai_responses(it, model=None, provider=provider.openai):
    "Collect a Delta stream, yielding incremental chunks and a final StreamSummary."
    async for d in it:
        if d.text:     yield {'text': d.text}
        if d.thinking: yield {'thinking': d.thinking}
    if d.raw['type'] == 'response.completed':
        yield normalize_openai_response(d.raw['response'], model, provider=provider)

#### Text

In [ ]:
mn,inp = 'gpt-4o-mini','Hi!'
resp = await oai_cli.responses.create_response(model=mn,input=inp)
comp = normalize_openai_response(resp, mn)

In [ ]:
resp = await oai_cli.responses.create_response(model=mn, input=inp, stream=True)
# async for o in  acollect_stream(norm_and_yield(resp, normalize_openai_response_event)): print_stream(o)
async for o in  _acollect_stream_openai_responses(norm_and_yield(resp, normalize_openai_response_event)): print_stream(o)

Hello

!

 How

 can

 I

 assist

 you

 today

?

In [ ]:
comp.message

Msg(role='assistant', content=[Part(type=<PartType.text: 'text'>, text='Hello! How can I assist you today?', data={'type': 'output_text', 'annotations': [], 'logprobs': [], 'text': 'Hello! How can I assist you today?'})], data=None)

In [ ]:
o.message

Msg(role='assistant', content=[Part(type=<PartType.text: 'text'>, text='Hello! How can I assist you today?', data={'type': 'output_text', 'annotations': [], 'logprobs': [], 'text': 'Hello! How can I assist you today?'})], data=None)

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.usage, o.usage)
test_eq(comp.tool_calls, o.tool_calls)

#### Thinking

In [ ]:
mn,inp = 'o4-mini','What is 31231231 * 12312?'
resp = await oai_cli.responses.create_response(model=mn,input=inp,reasoning={"effort": "low", "summary": "auto"})
comp = normalize_openai_response(resp, mn)

In [ ]:
resp = await oai_cli.responses.create_response(model=mn,input=inp,reasoning={"effort": "low", "summary": "auto"},stream=True)
# async for o in acollect_stream(norm_and_yield(resp, normalize_openai_response_event)): print_stream(o)
async for o in _acollect_stream_openai_responses(norm_and_yield(resp, normalize_openai_response_event)): print_stream(o)

🧠

🧠

🧠

🧠

🧠

...

518

,

916

,

072

In [ ]:
test_eq('thinking' in L(comp.message.content).attrgot('type'), True)
test_eq('thinking' in L(o.message.content).attrgot('type'), True)

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)

#### Tool call

In [ ]:
mn,inp = 'gpt-4o-mini','What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'
tools=[{"type": "function", **sch['function']}]
resp = await oai_cli.responses.create_response(model=mn,input=inp,tools=tools)
comp = normalize_openai_response(resp, mn)

In [ ]:
resp = await oai_cli.responses.create_response(model=mn, input=inp, tools=tools, stream=True)
# async for o in  acollect_stream(norm_and_yield(resp, normalize_openai_response_event)): print_stream(o)
async for o in _acollect_stream_openai_responses(norm_and_yield(resp, normalize_openai_response_event)): print_stream(o)

In [ ]:
comp.tool_calls

[ToolCall(id='fc_07e79637b65d510c0069d66ff214c48196b5e751ed4d144a21', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_DAJIJCaYpEhIjo9Uj9payJqN'}),
 ToolCall(id='fc_07e79637b65d510c0069d66ff214d481968f1cc2376401e205', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_GlfRTq3CNf8ltWxNpcGyW5XC'})]

In [ ]:
o.tool_calls

[ToolCall(id='fc_0637c35cbc783b2e0069d7bad21bb48193b1e426c977e52b5a', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_dKULgc7ZwOThLi6YVhw2CMEv'}),
 ToolCall(id='fc_0637c35cbc783b2e0069d7bad21bc48193a6668686b57b1802', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_xldlxaK8iqg5vE8DRXGCRJEX'})]

In [ ]:
comp.message.content

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'fc_07e79637b65d510c0069d66ff214c48196b5e751ed4d144a21', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_DAJIJCaYpEhIjo9Uj9payJqN'}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'fc_07e79637b65d510c0069d66ff214d481968f1cc2376401e205', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_GlfRTq3CNf8ltWxNpcGyW5XC'})]

In [ ]:
o.message.content

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'fc_0637c35cbc783b2e0069d7bad21bb48193b1e426c977e52b5a', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_dKULgc7ZwOThLi6YVhw2CMEv'}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'fc_0637c35cbc783b2e0069d7bad21bc48193a6668686b57b1802', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_xldlxaK8iqg5vE8DRXGCRJEX'})]

In [ ]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(set(comp.message.content[0].data.keys()), set(o.message.content[0].data.keys()))
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.usage, o.usage)

#### Web search

In [ ]:
mn,inp = 'gpt-4o-mini','What is the weather in Istanbul today?'
tools=[{"type": "web_search_preview"}]
resp = await oai_cli.responses.create_response(model=mn,input=inp,tools=tools)
comp = normalize_openai_response(resp, mn)

In [ ]:
resp = await oai_cli.responses.create_response(model=mn, input=inp, tools=tools, stream=True)
# async for o in  acollect_stream(norm_and_yield(resp, normalize_openai_response_event)): print_stream(o)
async for o in _acollect_stream_openai_responses(norm_and_yield(resp, normalize_openai_response_event)): print_stream(o)

As

 of

5

:

42

...

 a

 local

 weather

 service

.

In [ ]:
comp.tool_calls

[ToolCall(id='ws_054bf0148beb23320069d7b85f2f608190af10f01fff65154d', name='web_search', arguments={'type': 'search', 'queries': ['Istanbul weather today'], 'query': 'Istanbul weather today'}, server=True, extra={'type': 'web_search_call', 'status': 'completed', 'action': {'type': 'search', 'queries': ['Istanbul weather today'], 'query': 'Istanbul weather today'}})]

In [ ]:
o.tool_calls

[ToolCall(id='ws_0b8a28b15eee6eda0069d7bae30d088197a1170a678e5b03e1', name='web_search', arguments={'type': 'search', 'queries': ['Istanbul weather today'], 'query': 'Istanbul weather today'}, server=True, extra={'type': 'web_search_call', 'status': 'completed', 'action': {'type': 'search', 'queries': ['Istanbul weather today'], 'query': 'Istanbul weather today'}})]

In [ ]:
comp.finish_reason, o.finish_reason

('stop', 'stop')

In [ ]:
comp.usage, o.usage

(Usage(prompt_tokens=309, completion_tokens=435, total_tokens=744, raw={'input_tokens': 309, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 435, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 744, 'server_tool_use': {'web_search_call': 1}}),
 Usage(prompt_tokens=309, completion_tokens=345, total_tokens=654, raw={'input_tokens': 309, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 345, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 654, 'server_tool_use': {'web_search_call': 1}}))

### OpenAI Chat Events

`PartAccum` part accumulator is used across openai chat, anthropic and gemini streaming events to collate the parts during parts.

In [ ]:
#| export
@dataclass
class PartAccum:
    parts: dict[Part|ToolCall] = field(default_factory=dict)
    tool_calls: list[ToolCall] = field(default_factory=list)
    
    def append(self, typ, index, txt=None, **tc_kwargs):
        'Create and accumulate same type sequential parts'
        if index not in self.parts: 
            if typ==PartType.tool_use: self.parts[index] = ToolCall(**tc_kwargs)
            else:                      self.parts[index] = Part(type=typ, text=txt)
        else:
            if typ==PartType.tool_use:
                    new_args = tc_kwargs.get('arguments', '')
                    cur_args = self.parts[index].arguments
                    if isinstance(new_args, str) and isinstance(cur_args, str): self.parts[index].arguments += new_args
                    elif isinstance(new_args, str) and isinstance(cur_args, dict): self.parts[index].arguments = new_args
                    else: self.parts[index].arguments = new_args
            else:                      self.parts[index] = Part(type=typ, text=(self.parts[index].text or '') + txt)
    
    def finalize(self):
        for idx,tc in self.parts.items():
            if isinstance(tc, ToolCall):
                if isinstance(tc.arguments, str): tc.arguments = json.loads(tc.arguments)
                self.tool_calls.append(tc)
                self.parts[idx] = Part(type=PartType.tool_use, data=dict(id=tc.id, name=tc.name, arguments=tc.arguments, **tc.extra))
        # Merge consecutive same-type text/thinking parts
        merged = []
        for p in self.parts.values():
            if merged and merged[-1].type == p.type and p.type in (PartType.text, PartType.thinking):
                merged[-1] = Part(type=p.type, text=(merged[-1].text or '') + (p.text or ''))
            else: merged.append(p)
        self.parts = merged

A generic `_acollect_stream` function that yields thinking and text (if needed we can yield tool calls to), and collates parts while keeping the order. A custom `index_fn` is used to resolve the index that the current `Delta` event belongs to.

In [ ]:
#| export
async def _acollect_stream(it, index_fn, model=None, provider=None):
    "Collect a Delta stream, yielding incremental chunks and a final Completion."
    part_accum = PartAccum()
    deltas = []
    typ, last_typ, last_idx, last_idx = None, None, None, None
    async for d in it:
        if d.text:     
            yield {'text': d.text}
            typ = PartType.text
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.append(typ, idx, d.text)
        if d.thinking: 
            yield {'thinking': d.thinking}
            typ = PartType.thinking
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.append(typ, idx, d.thinking)
        for tc in d.tool_calls:
            args = tc.arguments.get('_delta', '') if '_delta' in tc.arguments else tc.arguments
            tc_kwargs = dict(id=tc.id, name=tc.name, arguments=args, server=tc.server, extra=tc.extra)
            typ = PartType.tool_use
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.append(typ, idx, **tc_kwargs)
        if d.server_tool_result: 
            typ = PartType.server_tool_result
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.parts[idx] = Part(type=typ, data=d.server_tool_result)
        if d.refusal:
            typ = PartType.refusal
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.append(typ, idx, d.refusal)        
        if d.finish_reason: fin = d.finish_reason
        if d.usage: usg = d.usage
        last_typ = typ
        deltas.append(d)
    part_accum.finalize()
    fin = canon_finish(fin, provider, part_accum.tool_calls) # need to canon one more time with accum'd tool calls
    yield Completion(d.raw.get('model', model),
            message=Msg(role="assistant", content=part_accum.parts),
            finish_reason=fin,
            usage=usg,
            tool_calls=part_accum.tool_calls,
            provider=provider,
            raw={'deltas':deltas})

In [ ]:
#| export
def openai_chat_index_fn(d, typ, last_typ, last_idx):
    'Returns accumulation index for current delta and updated last idx'
    if d.tool_calls: 
        tc_idx = nested_idx(d.tool_calls, 0, 'extra', 'index')
        return f"tool_{tc_idx}", last_idx
    if not (last_typ or last_idx): return 0,0
    if typ == last_typ: return last_idx, last_idx
    return last_idx + 1, last_idx + 1

In [ ]:
#| export
_acollect_stream_openai_chat = partial(_acollect_stream, index_fn=openai_chat_index_fn, provider=provider.openai_chat)

In [ ]:
#| export
@delegates(model_and_provider)
def normalize_openai_chat_delta(ev, **kwargs):
    "Normalize a chat completion stream event."
    kwargs['model'],kwargs['provider'] = kwargs.get('model',None), kwargs.get('provider',None)
    # usage always arrives as a single final event with choices: []
    if not (choices := ev.get("choices", [])): return Delta(usage=usage_from_openai(ev), raw=ev, **kwargs)
    # finish_reason arrives in its own dedicated chunk (empty delta, non-null finish_reason)
    if fin := nested_idx(ev, 'choices', 0, 'finish_reason'): return Delta(finish_reason=fin, raw=ev, **kwargs)
    # repurposed the common function
    tcs = openai_chat_tool_calls(ev, delta=True)
    dlt = nested_idx(choices, 0, 'delta')
    return Delta(text=dlt.get('content'), thinking=dlt.get('reasoning_content'), refusal=dlt.get('refusal'), tool_calls=tcs, raw=ev, **kwargs)

#### Text

In [ ]:
mn,msgs = 'gpt-4o-mini',[{"role": "user", "content": "Say hi"}]
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=msgs)
comp = normalize_openai_chat_completion(resp, mn)

In [ ]:
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=msgs,stream=True,stream_options={"include_usage": True})
async for o in _acollect_stream_openai_chat(norm_and_yield(resp, normalize_openai_chat_delta)): print_stream(o)

Hi

!

 How

 can

 I

 assist

 you

 today

?

In [ ]:
comp.message

Msg(role='assistant', content=[Part(type='text', text='Hi! How can I assist you today?', data=None)], data=None)

In [ ]:
o.message

Msg(role='assistant', content=[Part(type=<PartType.text: 'text'>, text='Hi! How can I assist you today?', data=None)], data=None)

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.usage, o.usage)
test_eq(comp.tool_calls, o.tool_calls)

#### Thinking

OpenAI's Chat Completions API doesn't expose reasoning content — `reasoning_tokens` appear in usage but no `reasoning_content` field is returned. We use Kimi (`kimi-k2.5`) via Moonshot's OpenAI-compatible API to test thinking parts in chat completions streaming.

**NB**: Kimi's thinking is controlled via a `thinking` body param (not `reasoning_effort`). Use `_body={"thinking": {"type": "disabled"}}` to disable it, or `_body={"thinking": {"type": "enabled"}}` to enable. Since `thinking` isn't in the OpenAI spec, `fastspec` requires the `_body` escape hatch.

Kimi's thinking is binary — enabled (default) or disabled. There's no `reasoning_effort` level (low/medium/high) or `budget_tokens` like OpenAI/Anthropic.

**TODO**: Expose `_body`/`native` escape hatches to users in the high-level API so provider-specific params (like Kimi's `thinking`) can be passed through without needing to drop down to raw `fastspec` calls.

In [ ]:
mn,msgs = 'kimi-k2.5',[{"role": "user", "content": 'What is 31231231 * 12312?'}]
resp = await kimi_cli.chat.create_chat_completion(model=mn,messages=msgs,reasoning_effort='low')
comp = normalize_openai_chat_completion(resp, mn)

In [ ]:
resp = await kimi_cli.chat.create_chat_completion(model=mn,messages=msgs,stream=True,stream_options={"include_usage": True})
async for o in _acollect_stream_openai_chat(norm_and_yield(resp, normalize_openai_chat_delta)): print_stream(o)

🧠

🧠

🧠

🧠

🧠

...

,

916

,

072

**

In [ ]:
test_eq('thinking' in L(comp.message.content).attrgot('type'), True)
test_eq('thinking' in L(o.message.content).attrgot('type'), True)

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)

#### Tool call

In [ ]:
mn = 'gpt-4o-mini'
msgs = [{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}]
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=msgs, tools=[sch])
comp = normalize_openai_chat_completion(resp, mn)

In [ ]:
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=msgs, tools=[sch], stream=True, stream_options={"include_usage": True})
async for o in _acollect_stream_openai_chat(norm_and_yield(resp, normalize_openai_chat_delta)): print_stream(o)

In [ ]:
comp.tool_calls

[ToolCall(id='call_HqTEPRlvekmBlxpzrhHGA6rm', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'type': 'function'}),
 ToolCall(id='call_QQajmcG0ACMnwixiTRgcLNOl', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'type': 'function'})]

In [ ]:
o.tool_calls

[ToolCall(id='call_lH6XatachWXyyOIOHeSMr6Mh', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'index': 0, 'type': 'function'}),
 ToolCall(id='call_ZMbiQm8Hah7xrNCNnUhUP9w5', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'index': 1, 'type': 'function'})]

In [ ]:
comp.message.content

[Part(type='tool_use', text=None, data={'id': 'call_HqTEPRlvekmBlxpzrhHGA6rm', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'type': 'function'}),
 Part(type='tool_use', text=None, data={'id': 'call_QQajmcG0ACMnwixiTRgcLNOl', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'type': 'function'})]

In [ ]:
o.message.content

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'call_lH6XatachWXyyOIOHeSMr6Mh', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'index': 0, 'type': 'function'}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'call_ZMbiQm8Hah7xrNCNnUhUP9w5', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'index': 1, 'type': 'function'})]

In [ ]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

(Usage(prompt_tokens=66, completion_tokens=52, total_tokens=118, raw={'prompt_tokens': 66, 'completion_tokens': 52, 'total_tokens': 118, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}),
 Usage(prompt_tokens=66, completion_tokens=52, total_tokens=118, raw={'prompt_tokens': 66, 'completion_tokens': 52, 'total_tokens': 118, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}))

### Anthropic Messages Events

In [ ]:
#| export
def normalize_anthropic_event(ev, **kwargs):
    kwargs['model'],kwargs['provider'] = kwargs.get('model',None), kwargs.get('provider',None)
    typ = ev.get("type")
    text, thinking, tcs = None, None, [] 
    if typ == "content_block_start":
        cb = ev.get("content_block", {})
        if cb.get("type", "").endswith("_tool_result"): return Delta(server_tool_result=cb, raw=ev, **kwargs)
        if tc := anthropic_tool_call(cb): tcs = [tc]
    elif typ == "content_block_delta":
        d = ev.get("delta", {})
        dtyp = d.get("type")
        if dtyp == "text_delta": text = d.get("text")
        elif dtyp == "thinking_delta": thinking = d.get("thinking")
        elif dtyp == "input_json_delta":
            tcs = [ToolCall(id=str(ev.get("index", "")), name="", arguments={"_delta": d.get("partial_json", '')})]
    elif typ == "message_delta":
        fin = canon_finish(nested_idx(ev, 'delta', 'stop_reason'), 'anthropic')
        return Delta(finish_reason=fin, usage=usage_from_anthropic(ev), raw=ev, **kwargs)
    elif typ == "error": raise api_error_from_event(ev, provider="anthropic", endpoint="messages.stream")
    return Delta(text=text, thinking=thinking, tool_calls=tcs, raw=ev, **kwargs)

In [ ]:
#| export
def anthropic_index_fn(d, typ, last_typ, last_idx):
    'Returns accumulation index for current delta and updated last idx'
    return nested_idx(d, 'raw', 'index'), None

In [ ]:
#| export
_acollect_stream_anthropic = partial(_acollect_stream, index_fn=anthropic_index_fn, provider=provider.anthropic)

#### Thinking

In [ ]:
mn,msgs = 'claude-sonnet-4-20250514', [{"role": "user", "content": "Say hi"}]
think,hdrs = {"type": "enabled", "budget_tokens": 5000}, {"anthropic-version": "2023-06-01"}
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, max_tokens=8000,
                                            thinking=think, headers=hdrs)
comp = normalize_anthropic_message(resp, mn)

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn,messages=msgs,max_tokens=8000,thinking=think, headers=hdrs,stream=True)
async for o in _acollect_stream_anthropic(norm_and_yield(resp, normalize_anthropic_event)): print_stream(o)

🧠

🧠

🧠

Hi there! How are you doing today?

In [ ]:
comp.message

Msg(role='assistant', content=[Part(type=<PartType.thinking: 'thinking'>, text='The user is asking me to say hi, which is a simple and friendly greeting. I should respond in a warm and welcoming way.', data={'type': 'thinking', 'thinking': 'The user is asking me to say hi, which is a simple and friendly greeting. I should respond in a warm and welcoming way.', 'signature': 'ErwCCmIIDBgCKkCIpOHL0OyUJ6GZPDJDV+loYiyMIRrCsZcQRzZo30WQChS1jrJbVzWFAfUezlsxhp2g08nYn0u7twzVODfcCYVkMhhjbGF1ZGUtc29ubmV0LTQtMjAyNTA1MTQ4ABIMBnYuHNEVcvb8e8zxGgzkm7oqiVgnIuFAUXEiMP9+nYRBq5nzLNAcEtn2nSRa+NIseIzomyzgtvYJ2eC4ef0WDP1L4SwmbZAcEzrRkCqHAaA233o9ADnZb5BjG7F0HcsPMeO3FRcNJEwIujIf0gQjc2mCTxnwx8UYXxBprqbAtEQ8/plNzsYBiWNjyfZDoHBZUwnJem44embFMd99B4hhmuc3FsIgtO4ow4W/RxQnhGlHTdIXgLlZ99+nOlKiJ4SaWsbXbuYWLNWeue7tvTkxlfLhxsRZbRgB'}), Part(type=<PartType.text: 'text'>, text='Hi! How are you doing today? Is there anything I can help you with?', data={'type': 'text', 'text': 'Hi! How are you doing today? Is there anything I

In [ ]:
o.message

Msg(role='assistant', content=[Part(type=<PartType.thinking: 'thinking'>, text='The user is asking me to say hi, which is a simple greeting. This is a straightforward request that I can respond to in a friendly and polite manner.', data=None), Part(type=<PartType.text: 'text'>, text='Hi there! How are you doing today?', data=None)], data=None)

In [ ]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)

#### Tool call

In [ ]:
mn,msgs = 'claude-sonnet-4-20250514', [{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}]
think,hdrs = {"type": "enabled", "budget_tokens": 5000}, {"anthropic-version": "2023-06-01"}
tools = [{"name": "simple_add", "description": "add numbers", "input_schema": sch['function']['parameters']}]
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, tools=tools, max_tokens=8000, thinking=think, headers=hdrs)
comp = normalize_anthropic_message(resp, mn)

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, tools=tools, max_tokens=8000, thinking=think, headers=hdrs, stream=True)
async for o in _acollect_stream_anthropic(norm_and_yield(resp, normalize_anthropic_event)): print_stream(o)

🧠

🧠

🧠

🧠

🧠

🧠

I'll calculate both additions for you using the simple_add tool in parallel.

In [ ]:
comp.tool_calls

[ToolCall(id='toolu_011eaEZkzBLtZQpHT5BfpX7j', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'caller': {'type': 'direct'}}),
 ToolCall(id='toolu_01UutnRB51PiKHZEGCtPFiCg', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'caller': {'type': 'direct'}})]

In [ ]:
o.tool_calls

[ToolCall(id='toolu_016rVuo7JtGaDnRPWvvmRAKb', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'caller': {'type': 'direct'}}),
 ToolCall(id='toolu_01Jpvs7DMGusSkYqxmE59dsF', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'caller': {'type': 'direct'}})]

In [ ]:
comp.message.content

[Part(type=<PartType.thinking: 'thinking'>, text='The user wants me to calculate 3 + 5 and 10 + 20 using the simple_add tool. They specifically mentioned doing this "in parallel", which means I should make both function calls at the same time rather than sequentially.\n\nLooking at the function schema:\n- Function name: simple_add\n- Required parameters: a (integer), b (integer)\n\nFor the first calculation: 3 + 5\n- a = 3\n- b = 5\n\nFor the second calculation: 10 + 20\n- a = 10\n- b = 20\n\nI have all the required parameters, so I can proceed with both function calls in parallel.', data={'type': 'thinking', 'thinking': 'The user wants me to calculate 3 + 5 and 10 + 20 using the simple_add tool. They specifically mentioned doing this "in parallel", which means I should make both function calls at the same time rather than sequentially.\n\nLooking at the function schema:\n- Function name: simple_add\n- Required parameters: a (integer), b (integer)\n\nFor the first calculation: 3 + 5\n-

In [ ]:
o.message.content

[Part(type=<PartType.thinking: 'thinking'>, text="The user is asking for two addition operations:\n1. 3 + 5\n2. 10 + 20\n\nThey specifically want me to use the simple_add tool and do it in parallel. I have the simple_add function available which takes two integer parameters 'a' and 'b'. I can make both function calls in the same function_calls block to execute them in parallel.\n\nFor the first calculation: a=3, b=5\nFor the second calculation: a=10, b=20", data=None),
 Part(type=<PartType.text: 'text'>, text="I'll calculate both additions for you using the simple_add tool in parallel.", data=None),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'toolu_016rVuo7JtGaDnRPWvvmRAKb', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'caller': {'type': 'direct'}}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'toolu_01Jpvs7DMGusSkYqxmE59dsF', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'caller': {'type': 'direct'}})]

In [ ]:
test_eq(comp.tool_calls[0].server, False)
for tc in comp.tool_calls: tc.id = '123'
for tc in o.tool_calls:    tc.id = '123'
test_eq(comp.tool_calls, o.tool_calls)

In [ ]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

(Usage(prompt_tokens=447, completion_tokens=297, total_tokens=744, raw={'input_tokens': 447, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 297, 'service_tier': 'standard', 'inference_geo': 'not_available'}),
 Usage(prompt_tokens=447, completion_tokens=260, total_tokens=707, raw={'input_tokens': 447, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 260}))

#### Web search

In [ ]:
mn,msgs = 'claude-sonnet-4-20250514', [{"role": "user", "content": "Use web search to get the weather in Istanbul?"}]
think,hdrs = {"type": "enabled", "budget_tokens": 5000}, {"anthropic-version": "2023-06-01"}
tools=[{"type": "web_search_20250305", "name": "web_search"}]
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, tools=tools, max_tokens=8000, thinking=think, headers=hdrs)
comp = normalize_anthropic_message(resp, mn)

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, tools=tools, max_tokens=8000, thinking=think, headers=hdrs, stream=True)
async for o in _acollect_stream_anthropic(norm_and_yield(resp, normalize_anthropic_event)): print_stream(o)

🧠

🧠

🧠

🧠

🧠

...

High temperature around 49°F (

9°C), with winds from the north at 5-10 mph and a

 30% chance of rain



The weather appears to be cool and overcast with some possibility of light showers throughout

 the day, typical spring weather for Istanbul.

In [ ]:
L(comp.message.content).attrgot('type'), L(o.message.content).attrgot('type')

([<PartType.thinking: 'thinking'>, <PartType.tool_use: 'tool_use'>, <PartType.server_tool_result: 'server_tool_result'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>],
 [<PartType.thinking: 'thinking'>, <PartType.tool_use: 'tool_use'>, <PartType.server_tool_result: 'server_tool_result'>, <PartType.text: 'text'>])

In [ ]:
test_eq(comp.tool_calls[0].server, True)
for tc in comp.tool_calls: tc.id = '123'
for tc in o.tool_calls:    tc.id = '123'
test_eq(comp.tool_calls, o.tool_calls)

In [ ]:
test_eq('server_tool_result' in L(comp.message.content).attrgot('type'), True)
test_eq('server_tool_result' in L(o.message.content).attrgot('type'), True)

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

(Usage(prompt_tokens=11752, completion_tokens=622, total_tokens=12374, raw={'input_tokens': 11752, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 622, 'service_tier': 'standard', 'inference_geo': 'not_available', 'server_tool_use': {'web_search_requests': 1, 'web_fetch_requests': 0}}),
 Usage(prompt_tokens=11783, completion_tokens=507, total_tokens=12290, raw={'input_tokens': 11783, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 507, 'server_tool_use': {'web_search_requests': 1, 'web_fetch_requests': 0}}))

**TODO**: Check why `cache_creation` doesn't exist in streaming

### Gemini Generate Events

In [ ]:
#| export
@delegates(model_and_provider)
def normalize_gemini_event(ev, **kwargs):
    "Normalize Gemini stream event into Delta."
    kwargs['model'],kwargs['provider'] = kwargs.get('model',None), kwargs.get('provider',None)
    cand = nested_idx(ev, 'candidates', 0) or {}
    finish_reason = canon_finish(cand.get("finishReason"), 'gemini')
    parts = nested_idx(cand, 'content', 'parts') or []
    thinking = "".join(p.get("text","") for p in parts if p.get("thought") and "text" in p)
    txt = "".join(p.get("text","") for p in parts if not p.get("thought") and "text" in p)
    tcs = gemini_tool_calls(ev)
    return Delta(text=txt, thinking=thinking, tool_calls=tcs, finish_reason=finish_reason,
                 usage=usage_from_gemini(ev), raw=ev, **kwargs)

In [ ]:
#| export
def gemini_index_fn(d, typ, last_typ, last_idx):
    'Returns accumulation index for current delta and updated last idx'
    if not (last_typ or last_idx): return 0,0
    return last_idx + 1, last_idx + 1

In [ ]:
#| export
_acollect_stream_gemini = partial(_acollect_stream, index_fn=gemini_index_fn, provider=provider.gemini)

#### Text

In [ ]:
mn,inp = 'models/gemini-3-flash-preview',[{"role": "user", "parts": [{"text": "Hi how are you?"}]}]
resp = await gem_cli.models.generate_content(model=mn,contents=inp)
comp = normalize_gemini_generate(resp, mn)

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=inp, stream=True, _query={"alt": "sse"})
async for o in _acollect_stream_gemini(norm_and_yield(resp, normalize_gemini_event)): print_stream(o)

I'm doing great, thank you for asking! How are you doing today?

 Is there anything I can help you with?

In [ ]:
comp.message

Msg(role='assistant', content=[Part(type=<PartType.text: 'text'>, text="I'm doing well, thank you for asking! How are you doing today? Is there anything I can help you with?", data={'text': "I'm doing well, thank you for asking! How are you doing today? Is there anything I can help you with?", 'thoughtSignature': 'EqkDCqYDAb4+9vuqsyJ/zM8IR/8jeL+yQ2JLCdMDdIxF7j9SM3+wUOWWDyn6GNpmHRr8LUW86buAYz0cV8V48j9kS2dQpY87tFLAU8CLcGfLc10E4NwpuRd38AaXeFVOyqwuZKg9Iq7g/0bNLZK4mqQvmjfeUmeLzViRTMTkbIK59MmffSNgANRFKFbL6qPNZpvgh8NuAmwc3AuYyFM2rjhrRX5Wmd7dt9x+Hrv/+xrWTl2UZp1WAWWriaJO32KTyTcUYm2CqwciUVc543m4pjwGlubClTgrzDoxJ5kyAJ9a3ykNaTa5/WZXW8/QfR2U+2IWAb/n9OF14c80Z5v0qKo2Xmou7BmDB039ocVohjmT9NfH2sxbZOo0fSedJrrc6qW09skiL/3dFGlbVnSRl76eMH5BKmdVhfTIPhGpoV7wsv6hL1mwqVojHzJVCUcQnILXEDf+fEFiuTFfTG87OdXUu4C9F0gSMKTmDstxB9IBLJnYuv0PiNrxU1/S6Bwt2rcoI//Kd3RTxFW/U3Cc4FbN//J67bTl4NnOKRkt/THEBsBMJ/DilRcKQ9Y='})], data=None)

In [ ]:
o.message

Msg(role='assistant', content=[Part(type=<PartType.text: 'text'>, text="I'm doing great, thank you for asking! How are you doing today? Is there anything I can help you with?", data=None)], data=None)

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)
comp.usage, o.usage

(Usage(prompt_tokens=6, completion_tokens=26, total_tokens=132, raw={'promptTokenCount': 6, 'candidatesTokenCount': 26, 'totalTokenCount': 132, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}], 'thoughtsTokenCount': 100}),
 Usage(prompt_tokens=6, completion_tokens=26, total_tokens=188, raw={'promptTokenCount': 6, 'candidatesTokenCount': 26, 'totalTokenCount': 188, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}], 'thoughtsTokenCount': 156}))

#### Thinking

- **Thinking always happens internally** for Gemini 3/2.5 models — token cost visible in `thoughtsTokenCount` in usage metadata
- **Thought text in response** only returned when `include_thoughts: True` is explicitly set in `generation_config.thinking_config` — defaults to `false`
- **`thoughtSignature`** is always returned regardless of `includeThoughts` — required for multi-turn context continuity. Missing signatures in function calling result in a 400 error; for text/chat, omitting them degrades reasoning quality
- **`thinking_level`** controls reasoning depth: `high` (default for Gemini 3), `medium`, `low`, `minimal` (Flash only)

In [ ]:
mn,inp = 'models/gemini-3-flash-preview',[{"role": "user", "parts": [{"text": "What is 31231231 * 12312?"}]}]
resp = await gem_cli.models.generate_content(model=mn,contents=inp,generation_config={"thinking_config": {"include_thoughts": True}})
comp = normalize_gemini_generate(resp, mn)

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=inp, generation_config={"thinking_config": {"include_thoughts": True}}, stream=True, _query={"alt": "sse"})
async for o in _acollect_stream_gemini(norm_and_yield(resp, normalize_gemini_event)): print_stream(o)

🧠

🧠

🧠

To calculate **31,231,231 × 

12,312**, we can break the multiplication down using the distributive property:

$31,231,

...

  312,312,310,000
+  62,462

,462,000
+   9,369,369,300


+     312,312,310
+      62,462,

462
_________________
**384,518,916,072**

The

 answer is **384,518,916,072**.

In [ ]:
L(comp.message.content).attrgot('type'), #comp.message.content

([<PartType.thinking: 'thinking'>, <PartType.text: 'text'>],)

In [ ]:
L(o.message.content).attrgot('type')#, o.message.content

[<PartType.thinking: 'thinking'>, <PartType.text: 'text'>]

In [ ]:
test_eq('thinking' in L(comp.message.content).attrgot('type'), True)
test_eq('thinking' in L(o.message.content).attrgot('type'), True)

In [ ]:
L(comp.message.content).attrgot('type'), L(o.message.content).attrgot('type')

([<PartType.thinking: 'thinking'>, <PartType.text: 'text'>],
 [<PartType.thinking: 'thinking'>, <PartType.text: 'text'>])

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)
comp.usage, o.usage

(Usage(prompt_tokens=20, completion_tokens=430, total_tokens=1599, raw={'promptTokenCount': 20, 'candidatesTokenCount': 430, 'totalTokenCount': 1599, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 20}], 'thoughtsTokenCount': 1149}),
 Usage(prompt_tokens=20, completion_tokens=382, total_tokens=2437, raw={'promptTokenCount': 20, 'candidatesTokenCount': 382, 'totalTokenCount': 2437, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 20}], 'thoughtsTokenCount': 2035}))

**TODO:** Thought signature is correctly preserved in Completion part data however streaming is missing it. I'm thinking if we should ignore it for now and stick to injecting dummy thought signatures for gemini as needed.

#### Tool call

**TODO:** Gemini `thoughtSignature` handling:
- **Completions**: correctly preserved in `Part.data` for both text and tool call parts
- **Streaming**: missing — not yet captured from stream events
- **Required?** Yes for function call parts (Gemini 3 returns 400 if missing); recommended but not required for text parts (degrades quality) 
- **Cross-provider**: Gemini accepts dummy signatures `"context_engineering_is_the_way_to_go"` or `"skip_thought_signature_validator"` — inject these on function call parts when crossing from non-Gemini providers
- **Placement**: first FC part gets the signature for parallel calls; each FC part gets one for sequential calls; last part gets it for non-FC responses

In [ ]:
inp = [{"role": "user", "parts": [{"text": 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]}]
resp = await gem_cli.models.generate_content(model=mn, contents=inp, tools=[{"functionDeclarations": [sch['function']]}])
comp = normalize_gemini_generate(resp, mn)

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=inp, tools=[{"functionDeclarations": [sch['function']]}], stream=True, _query={"alt": "sse"})
async for o in _acollect_stream_gemini(norm_and_yield(resp, normalize_gemini_event)): print_stream(o)

In [ ]:
comp.tool_calls

[ToolCall(id='ph64b27g', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'thoughtSignature': 'EvACCu0CAb4+9vshNyjJn/UHIXFVEhmWXWHX+ZDWZEuzJQipewGfo9yDbSNJGUndutczwTe1rwt5FhDj0izLrQjbTmhnY9NRhvMyNAEt+lueSAZms08Wb96Bh0jjOIaGqgebVbYRIop7AR/knYqae8+TgEsNGhbYeP2D7gJPXoSUn2A40oX+sAPVcMreHE2f8KBSufhi05nzdrT3EFWrU2P/0w1Z5OjSg+OiU7wIYMQyeb5OrASJQQseFMPfLdqCiUMGg5X8TLVwaqhHuNoIQonA9fexuOhjznQIbrOP3W54HcYSm0jsHUUX1uuR+Mot+4IwH0xJKKZLvA4yzn+dbonmjlNbz1LMUwLcnJh83h+F68rVwgOBJpl3Q8anhQAa1JKIvFYXPyyUczPJwWBHoM6R0OGoj99iG28VMnw2sl0ibyIEPgk6xhSB3UXmVDH59mWQxnI9eaOyxfvam8ZF9KqAg9HTgOfC1F67eZ4uvF+sNeM='}),
 ToolCall(id='k8wu5qdv', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={})]

In [ ]:
o.tool_calls

[ToolCall(id='vchmcu53', name='simple_add', arguments={'b': 5, 'a': 3}, server=False, extra={'thoughtSignature': 'EqUDCqIDAb4+9vvgiBWJe8rHQ4AsefEidQLNf3wESYqUssWiAripZiGOGQlKRycajyxuPB7kSU1sCX6zg/ULNAjMY9qklM9iUN38J2LqKwgUjJ6JC1grDOj+VxBT4DuB+2NZ1BKk5+pxMSp9/1vkArpfMN8ekYx8nk4izcpODjj846vfl8sKFu0sMHjDazItyTUrfPFniUVcX8aGbdMVjQQt5jh9DcFI5nBh8LD1hrlck29IRvwnL0OcbyndUDyKewMS8msM22dsYE4gJg0AqrLcCsKBb+arW9zLWQFcC85qEtCLs+EQMdk6pTSYQOrZf2KJZJHXW3kv11ioq8p81aSyq6xnAUSDs9og7azl5XKyL+nWXe5Zum3aCyjva4NEDpIotMvrW0wH1cXPaR7WPqxVg9qk/YNuGgRlZ2cQDrKLuge1/z9/vk9v4w4/t24sJzYJRdFH4RWgdTqmHXhFz78Q62hAQtHXHpWXnwHtrQ/yRyzdimy3DC2XHt9SbEDd4ULaJy/AI0EyyDJTFfZALh2F5tyLf7iK929aVChkAUUg6p3AsR95XQ=='}),
 ToolCall(id='7rwckpni', name='simple_add', arguments={'b': 20, 'a': 10}, server=False, extra={})]

In [ ]:
comp.message.content

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'ph64b27g', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'thoughtSignature': 'EvACCu0CAb4+9vshNyjJn/UHIXFVEhmWXWHX+ZDWZEuzJQipewGfo9yDbSNJGUndutczwTe1rwt5FhDj0izLrQjbTmhnY9NRhvMyNAEt+lueSAZms08Wb96Bh0jjOIaGqgebVbYRIop7AR/knYqae8+TgEsNGhbYeP2D7gJPXoSUn2A40oX+sAPVcMreHE2f8KBSufhi05nzdrT3EFWrU2P/0w1Z5OjSg+OiU7wIYMQyeb5OrASJQQseFMPfLdqCiUMGg5X8TLVwaqhHuNoIQonA9fexuOhjznQIbrOP3W54HcYSm0jsHUUX1uuR+Mot+4IwH0xJKKZLvA4yzn+dbonmjlNbz1LMUwLcnJh83h+F68rVwgOBJpl3Q8anhQAa1JKIvFYXPyyUczPJwWBHoM6R0OGoj99iG28VMnw2sl0ibyIEPgk6xhSB3UXmVDH59mWQxnI9eaOyxfvam8ZF9KqAg9HTgOfC1F67eZ4uvF+sNeM='}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'k8wu5qdv', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}})]

In [ ]:
o.message.content

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'vchmcu53', 'name': 'simple_add', 'arguments': {'b': 5, 'a': 3}, 'thoughtSignature': 'EqUDCqIDAb4+9vvgiBWJe8rHQ4AsefEidQLNf3wESYqUssWiAripZiGOGQlKRycajyxuPB7kSU1sCX6zg/ULNAjMY9qklM9iUN38J2LqKwgUjJ6JC1grDOj+VxBT4DuB+2NZ1BKk5+pxMSp9/1vkArpfMN8ekYx8nk4izcpODjj846vfl8sKFu0sMHjDazItyTUrfPFniUVcX8aGbdMVjQQt5jh9DcFI5nBh8LD1hrlck29IRvwnL0OcbyndUDyKewMS8msM22dsYE4gJg0AqrLcCsKBb+arW9zLWQFcC85qEtCLs+EQMdk6pTSYQOrZf2KJZJHXW3kv11ioq8p81aSyq6xnAUSDs9og7azl5XKyL+nWXe5Zum3aCyjva4NEDpIotMvrW0wH1cXPaR7WPqxVg9qk/YNuGgRlZ2cQDrKLuge1/z9/vk9v4w4/t24sJzYJRdFH4RWgdTqmHXhFz78Q62hAQtHXHpWXnwHtrQ/yRyzdimy3DC2XHt9SbEDd4ULaJy/AI0EyyDJTFfZALh2F5tyLf7iK929aVChkAUUg6p3AsR95XQ=='}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': '7rwckpni', 'name': 'simple_add', 'arguments': {'b': 20, 'a': 10}})]

In [ ]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(set(comp.message.content[0].data.keys()), set(o.message.content[0].data.keys()))
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

(Usage(prompt_tokens=77, completion_tokens=38, total_tokens=222, raw={'promptTokenCount': 77, 'candidatesTokenCount': 38, 'totalTokenCount': 222, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 77}], 'thoughtsTokenCount': 107}),
 Usage(prompt_tokens=77, completion_tokens=38, total_tokens=213, raw={'promptTokenCount': 77, 'candidatesTokenCount': 38, 'totalTokenCount': 213, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 77}], 'thoughtsTokenCount': 98}))

#### Web search

Gemini's Google Search is a transparent server-side tool — unlike Anthropic/OpenAI, it produces no `functionCall` parts or tool_use blocks. Search results appear as `groundingMetadata` on the candidate, detected by `usage_from_gemini` to set `server_tool_use: {"google_search": 1}`. The response content is plain text only, so `tool_calls` is always `[]`.

In [ ]:
inp = [{"role": "user", "parts": [{"text": "What is the weather in Istanbul today?"}]}]
resp = await gem_cli.models.generate_content(model=mn, contents=inp, tools=[{"googleSearch": {}}])
comp = normalize_gemini_generate(resp, mn)

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=inp, tools=[{"googleSearch": {}}], stream=True, _query={"alt": "sse"})
async for o in _acollect_stream_gemini(norm_and_yield(resp, normalize_gemini_event)): print_stream(o)

Today in Istanbul, the weather is **partly sunny** with comfortable spring temperatures.

Here

 is the detailed forecast for Monday, April 13, 2026:

*   **Temperature:** The

 high today is approximately **14°C (58°F)**, with an overnight low reaching **6°C (

43°F)**.
*   **Conditions:** It will be mostly sunny throughout the afternoon, becoming partly cloudy in the

 evening.
*   **Precipitation:** There is a very low chance of rain (around 3–10

%).
*   **Humidity:** The humidity level is moderate at about **45–53%**.
*   **

Wind:** Expect light breezes, typical for this time of year in the region.

Overall, it is a pleasant day for outdoor

 activities, though you may want a light jacket for the cooler morning and evening hours.

In [ ]:
comp.tool_calls

[]

In [ ]:
o.tool_calls

[]

In [ ]:
comp.message.content[0]

Part(type=<PartType.text: 'text'>, text='Today in Istanbul, it is **mostly sunny** with a current temperature of **10°C (50°F)**. \n\nThe forecast for the rest of the day, Thursday, April 9, 2026, includes the following:\n\n*   **Daytime:** Expect cloudy conditions with a high of **12°C (54°F)**. There is a 25% chance of rain during the day.\n*   **Nighttime:** Temperatures will drop to a low of **7°C (44°F)**, and light rain is expected (35% chance).\n*   **Conditions:** Humidity is around 67% currently, and it feels like 9°C (49°F) due to a slight breeze.\n\nIf you are heading out, it is a good idea to have a light jacket for the day and an umbrella if you plan to be out late this evening.', data={'text': 'Today in Istanbul, it is **mostly sunny** with a current temperature of **10°C (50°F)**. \n\nThe forecast for the rest of the day, Thursday, April 9, 2026, includes the following:\n\n*   **Daytime:** Expect cloudy conditions with a high of **12°C (54°F)**. There is a 25% chance of 

In [ ]:
o.message.content[0]

Part(type=<PartType.text: 'text'>, text='Today in Istanbul, the weather is **partly sunny** with comfortable spring temperatures.\n\nHere is the detailed forecast for Monday, April 13, 2026:\n\n*   **Temperature:** The high today is approximately **14°C (58°F)**, with an overnight low reaching **6°C (43°F)**.\n*   **Conditions:** It will be mostly sunny throughout the afternoon, becoming partly cloudy in the evening.\n*   **Precipitation:** There is a very low chance of rain (around 3–10%).\n*   **Humidity:** The humidity level is moderate at about **45–53%**.\n*   **Wind:** Expect light breezes, typical for this time of year in the region.\n\nOverall, it is a pleasant day for outdoor activities, though you may want a light jacket for the cooler morning and evening hours.', data=None)

In [ ]:
comp.finish_reason, o.finish_reason

('stop', 'stop')

In [ ]:
comp.usage, o.usage

(Usage(prompt_tokens=50, completion_tokens=198, total_tokens=537, raw={'promptTokenCount': 50, 'candidatesTokenCount': 198, 'totalTokenCount': 537, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 50}], 'thoughtsTokenCount': 289, 'server_tool_use': {'google_search': 1}}),
 Usage(prompt_tokens=67, completion_tokens=190, total_tokens=554, raw={'promptTokenCount': 67, 'candidatesTokenCount': 190, 'totalTokenCount': 554, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 67}], 'thoughtsTokenCount': 297, 'server_tool_use': {'google_search': 1}}))

### acollect_stream

In [ ]:
#| export
async def acollect_stream(it, model=None, provider=None):
    if provider == provider.openai:        fn = _acollect_stream_openai_responses
    elif provider == provider.openai_chat: fn = _acollect_stream_openai_chat
    elif provider == provider.anthropic:   fn = _acollect_stream_anthropic
    elif provider == provider.gemini:      fn = _acollect_stream_gemini
    else: raise ValueError(f"Unknown provider: {provider}")
    async for o in fn(it, model=model): yield o

In [ ]:
# OpenAI Responses
mn, inp = 'gpt-4o-mini', 'Say hi'
resp = await oai_cli.responses.create_response(model=mn, input=inp, stream=True)
async for o in acollect_stream(norm_and_yield(resp, normalize_openai_response_event), model=mn, provider=provider.openai): print_stream(o)
test_eq(o.message.content[0].type, PartType.text)
test_eq(o.finish_reason, 'stop')

Hi

 there

!

 How

 can

 I

 assist

 you

 today

?

In [ ]:
# OpenAI Chat
mn, msgs = 'gpt-4o-mini', [{"role": "user", "content": "Say hi"}]
resp = await oai_cli.chat.create_chat_completion(model=mn, messages=msgs, stream=True, stream_options={"include_usage": True})
async for o in acollect_stream(norm_and_yield(resp, normalize_openai_chat_delta), model=mn, provider=provider.openai_chat): print_stream(o)
test_eq(o.message.content[0].type, PartType.text)
test_eq(o.finish_reason, 'stop')

Hi

!

 How

 can

 I

 assist

 you

 today

?

In [ ]:
# Anthropic
mn, msgs = 'claude-sonnet-4-20250514', [{"role": "user", "content": "Say hi"}]
hdrs = {"anthropic-version": "2023-06-01"}
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, max_tokens=100, headers=hdrs, stream=True)
async for o in acollect_stream(norm_and_yield(resp, normalize_anthropic_event), model=mn, provider=provider.anthropic): print_stream(o)
test_eq(o.message.content[0].type, PartType.text)
test_eq(o.finish_reason, 'stop')

Hi there! How are you doing today

?

In [ ]:
# Gemini
mn, inp = 'models/gemini-3-flash-preview', [{"role": "user", "parts": [{"text": "Say hi"}]}]
resp = await gem_cli.models.stream_generate_content(model=mn, contents=inp, stream=True, _query={"alt": "sse"})
async for o in acollect_stream(norm_and_yield(resp, normalize_gemini_event), model=mn, provider=provider.gemini): print_stream(o)
test_eq(o.message.content[0].type, PartType.text)
test_eq(o.finish_reason, 'stop')

Hi! How can I help you today?

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()